# Stage 4: Human-in-the-Loop (HITL) & Time-Travel in LangGraph

**RepoLens Codebase Intelligence Engine**

This notebook is **100% self-contained and independent**. It demonstrates:
1. **Ambiguity Detection**: Identifying when a user's query matches multiple candidate files/symbols.
2. **LangGraph `interrupt()`**: Pausing graph execution and saving state to `MemorySaver` using a UUID `thread_id`.
3. **Resumption with `Command(resume=...)`**: Resuming execution seamlessly without re-running earlier steps.
4. **Time-Travel & State Rewind**: Inspecting previous checkpoints with `app.get_state_history()` and forking new conversation paths with `app.update_state()`.


In [1]:
# 1. Environment & Dependencies Setup
import os
import sys
import uuid
from pathlib import Path
from typing import List, Dict, Any, Optional, Literal
import dotenv
from pydantic import BaseModel, Field

# Load .env from project root
repo_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
dotenv.load_dotenv(repo_root / '.env')

print(f"[SETUP] Working Directory: {Path.cwd()}")
print(f"[SETUP] Groq API Key set: {bool(os.getenv('GROQ_API_KEY'))}")


[SETUP] Working Directory: e:\Learning\Campusx GenAI\RepoLens\notebooks
[SETUP] Groq API Key set: True


## 2. Model Factory & State Definition

In [2]:
from langchain_groq import ChatGroq
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

# Initialize Fast LLM for routing & ambiguity grading
PRIMARY_MODEL = os.getenv("PRIMARY_LLM_MODEL", "openai/gpt-oss-120b")
FAST_MODEL = os.getenv("FAST_ROUTER_MODEL", "openai/gpt-oss-20b")

llm = ChatGroq(
    model_name=FAST_MODEL,
    temperature=0.0,
    api_key=os.getenv("GROQ_API_KEY"),
)

print(f"[MODEL READY] Loaded Groq model: {FAST_MODEL}")


[MODEL READY] Loaded Groq model: openai/gpt-oss-20b


## 3. Ambiguity Analysis Schema & State

In [3]:
class AmbiguityAnalysis(BaseModel):
    """Determines if a query targets multiple distinct components."""
    is_ambiguous: bool = Field(description="True if the query refers to a name that exists in multiple distinct candidate files.")
    candidates: List[str] = Field(default_factory=list, description="List of candidate files or function paths.")
    clarification_question: str = Field(default="", description="Question prompting the user to disambiguate.")

class HitlAgentState(BaseModel):
    """Agent State with Ambiguity & Disambiguation Tracking."""
    messages: List[BaseMessage] = Field(default_factory=list)
    retrieved_candidates: List[str] = Field(default_factory=list)
    ambiguity_analysis: Optional[AmbiguityAnalysis] = None
    selected_entity: Optional[str] = None
    final_response: Optional[str] = None


## 4. Graph Nodes: Ambiguity Detection, `interrupt()`, and Generation

In [4]:
def retrieve_mock_candidates(state: HitlAgentState) -> Dict[str, Any]:
    """Simulates codebase symbol retrieval that yields ambiguous candidate files."""
    user_query = state.messages[-1].content.lower()
    print(f"\n🔍 [STEP 1: RETRIEVAL] Searching symbols for query: '{user_query}'")
    
    # Example scenario: query asks about 'authenticate'
    if "auth" in user_query or "token" in user_query or "login" in user_query:
        candidates = [
            "src/auth/jwt_handler.py (def authenticate(token: str))",
            "src/auth/oauth_client.py (def authenticate(code: str))",
            "src/api/v1/auth.py (def authenticate(request: Request))",
        ]
    elif "config" in user_query:
        candidates = [
            "src/config/app_config.py",
            "src/config/db_config.py",
        ]
    else:
        candidates = ["src/main.py"]
        
    return {"retrieved_candidates": candidates}


def analyze_ambiguity(state: HitlAgentState) -> Dict[str, Any]:
    """Evaluates if candidates require user clarification."""
    candidates = state.retrieved_candidates
    user_query = state.messages[-1].content
    
    if len(candidates) > 1 and not state.selected_entity:
        structured_llm = llm.with_structured_output(AmbiguityAnalysis)
        candidates_text = "\n".join([f"- {c}" for c in candidates])
        prompt = f"""User asked: "{user_query}"
Found multiple candidates in codebase:
{candidates_text}

Is the query ambiguous where the user needs to select which specific component they want?
If yes, set is_ambiguous=True, list candidates, and formulate a clear clarification question."""
        try:
            analysis = structured_llm.invoke(prompt)
        except Exception:
            analysis = AmbiguityAnalysis(
                is_ambiguous=True,
                candidates=candidates,
                clarification_question="I found multiple matches. Which one would you like to analyze?"
            )
        print(f"⚠️ [AMBIGUITY DETECTED] Is Ambiguous: {analysis.is_ambiguous}")
        return {"ambiguity_analysis": analysis}
    
    return {"ambiguity_analysis": AmbiguityAnalysis(is_ambiguous=False, candidates=[], clarification_question="")}


def hitl_disambiguate(state: HitlAgentState) -> Dict[str, Any]:
    """Pauses execution using LangGraph interrupt() until user supplies their choice."""
    analysis = state.ambiguity_analysis
    print(f"\n🛑 [HITL INTERRUPT] Pausing graph execution...")
    
    interrupt_payload = {
        "type": "disambiguation_prompt",
        "question": analysis.clarification_question,
        "options": analysis.candidates,
    }
    
    # LangGraph interrupt: Pauses graph, writes state to checkpointer, yields to caller
    user_selection = interrupt(interrupt_payload)
    
    print(f"▶️ [HITL RESUMED] Received user selection: '{user_selection}'")
    return {
        "selected_entity": user_selection,
        "messages": [AIMessage(content=f"Selected target: {user_selection}")]
    }


def generate_explanation(state: HitlAgentState) -> Dict[str, Any]:
    """Generates final grounded code explanation based on resolved selection."""
    target = state.selected_entity or state.retrieved_candidates[0]
    user_query = state.messages[0].content
    
    prompt = f"""Explain how `{target}` operates in response to the user's query: '{user_query}'.
Keep it concise, technical, and structured."""
    
    response = llm.invoke(prompt)
    print(f"\n💡 [GENERATION COMPLETE] Response length: {len(response.content)} chars")
    return {
        "final_response": response.content,
        "messages": [AIMessage(content=response.content)]
    }


## 5. Build & Compile StateGraph with Checkpointer

In [5]:
def route_after_analysis(state: HitlAgentState) -> Literal["hitl_disambiguate", "generate_explanation"]:
    analysis = state.ambiguity_analysis
    if analysis and analysis.is_ambiguous and not state.selected_entity:
        return "hitl_disambiguate"
    return "generate_explanation"

workflow = StateGraph(HitlAgentState)
workflow.add_node("retrieve_candidates", retrieve_mock_candidates)
workflow.add_node("analyze_ambiguity", analyze_ambiguity)
workflow.add_node("hitl_disambiguate", hitl_disambiguate)
workflow.add_node("generate_explanation", generate_explanation)

workflow.add_edge(START, "retrieve_candidates")
workflow.add_edge("retrieve_candidates", "analyze_ambiguity")
workflow.add_conditional_edges(
    "analyze_ambiguity",
    route_after_analysis,
    {
        "hitl_disambiguate": "hitl_disambiguate",
        "generate_explanation": "generate_explanation",
    }
)
workflow.add_edge("hitl_disambiguate", "generate_explanation")
workflow.add_edge("generate_explanation", END)

# Compile with MemorySaver checkpointer
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)
print("✅ [GRAPH COMPILED] StateGraph with interrupt() and MemorySaver ready!")


✅ [GRAPH COMPILED] StateGraph with interrupt() and MemorySaver ready!


## 6. Live Test: Triggering `interrupt()` and Resuming with `Command(resume=...)`

In [6]:
# 1. Create a unique UUID session thread
thread_id = f"session_{uuid.uuid4()}"
config = {"configurable": {"thread_id": thread_id}}
print(f"🔑 Session Thread ID: {thread_id}")

# 2. Submit an ambiguous query
initial_input = {
    "messages": [HumanMessage(content="How does the authenticate method work?")]
}

print("\n--- 🚀 RUNNING GRAPH (STEP 1) ---")
output = app.invoke(initial_input, config)

# 3. Check graph state for interruption
current_state = app.get_state(config)
print(f"\n📌 Next Node in Queue: {current_state.next}")

if current_state.tasks and hasattr(current_state.tasks[0], "interrupts"):
    interrupt_info = current_state.tasks[0].interrupts[0].value
    print(f"❓ Question: {interrupt_info.get('question')}")
    print("📋 Options to choose from:")
    for idx, opt in enumerate(interrupt_info.get("options", []), 1):
        print(f"   [{idx}] {opt}")


🔑 Session Thread ID: session_544a28ae-03de-488c-a7ae-877c44660f41

--- 🚀 RUNNING GRAPH (STEP 1) ---

🔍 [STEP 1: RETRIEVAL] Searching symbols for query: 'how does the authenticate method work?'


Deserializing unregistered type __main__.AmbiguityAnalysis from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'AmbiguityAnalysis')]


⚠️ [AMBIGUITY DETECTED] Is Ambiguous: True

🛑 [HITL INTERRUPT] Pausing graph execution...

📌 Next Node in Queue: ('hitl_disambiguate',)
❓ Question: I found multiple matches. Which one would you like to analyze?
📋 Options to choose from:
   [1] src/auth/jwt_handler.py (def authenticate(token: str))
   [2] src/auth/oauth_client.py (def authenticate(code: str))
   [3] src/api/v1/auth.py (def authenticate(request: Request))


## 7. Resuming Execution via `Command(resume=...)`

In [7]:
# User selects Option 1: JWT Authentication
chosen_option = "src/auth/jwt_handler.py (def authenticate(token: str))"
print(f"\n👉 User Selected: '{chosen_option}'")

print("\n--- 🚀 RESUMING GRAPH WITH Command(resume=...) ---")
final_state = app.invoke(Command(resume=chosen_option), config)

print("\n=================== FINAL AGENT ANSWER ===================")
print(final_state["final_response"])
print("==========================================================")



👉 User Selected: 'src/auth/jwt_handler.py (def authenticate(token: str))'

--- 🚀 RESUMING GRAPH WITH Command(resume=...) ---

🛑 [HITL INTERRUPT] Pausing graph execution...
▶️ [HITL RESUMED] Received user selection: 'src/auth/jwt_handler.py (def authenticate(token: str))'



💡 [GENERATION COMPLETE] Response length: 1852 chars

=================== FINAL AGENT ANSWER ===================
**`src/auth/jwt_handler.py – def authenticate(token: str)`**

| Step | What happens | Key details |
|------|--------------|-------------|
| 1. **Input** | Receives a JWT string (`token`). | Expected to be the `Authorization` header value minus the “Bearer ” prefix. |
| 2. **Decode & Verify** | Uses `jwt.decode()` (PyJWT) with:<br>• `SECRET_KEY` (from env or config)<br>• `ALGORITHM` (e.g., “HS256”) | Checks signature, algorithm, and optional `audience`/`issuer` claims. |
| 3. **Expiration Check** | `jwt.decode()` automatically raises `ExpiredSignatureError` if `exp` claim is past. | No manual `datetime` comparison needed. |
| 4. **Extract Claims** | Pulls standard claims (`sub`, `exp`, `iat`, `roles`, etc.) into a `payload` dict. | `sub` usually maps to user ID. |
| 5. **Return Value** | Returns the `payload` dict (or a `User` model if wrapped). | Caller can use `payload["sub

## 8. Time-Travel & State History Inspection

LangGraph automatically records every transition under the `thread_id`. We can:
1. Inspect all historical state checkpoints.
2. Rewind to a previous step.
3. Update state to fork execution.

In [8]:
print(f"🕒 Inspecting State History for Thread: {thread_id}\n")
history = list(app.get_state_history(config))

for idx, snapshot in enumerate(history):
    step_name = snapshot.next if snapshot.next else ('END',)
    checkpoint_id = snapshot.config['configurable'].get('checkpoint_id', 'init')
    print(f"[Snapshot {idx}] Checkpoint ID: {checkpoint_id[:12]}... | Next Action: {step_name}")
    if snapshot.values.get("selected_entity"):
        print(f"    -> Selected Entity: {snapshot.values['selected_entity']}")
    print("-" * 70)


🕒 Inspecting State History for Thread: session_544a28ae-03de-488c-a7ae-877c44660f41

[Snapshot 0] Checkpoint ID: 1f1b1a1c-175... | Next Action: ('END',)
    -> Selected Entity: src/auth/jwt_handler.py (def authenticate(token: str))
----------------------------------------------------------------------
[Snapshot 1] Checkpoint ID: 1f1b1a1c-0c8... | Next Action: ('generate_explanation',)
    -> Selected Entity: src/auth/jwt_handler.py (def authenticate(token: str))
----------------------------------------------------------------------
[Snapshot 2] Checkpoint ID: 1f1b1a1c-0c3... | Next Action: ('hitl_disambiguate',)
----------------------------------------------------------------------
[Snapshot 3] Checkpoint ID: 1f1b1a1c-058... | Next Action: ('analyze_ambiguity',)
----------------------------------------------------------------------
[Snapshot 4] Checkpoint ID: 1f1b1a1c-057... | Next Action: ('retrieve_candidates',)
----------------------------------------------------------------------
[

## 9. Time-Travel Forking: Rewind & Switch Choice to OAuth

We rewind to before generation, update the state to choose Option 2 (`oauth_client.py`), and re-run!

In [9]:
# Fork a new thread configuration based on time-travel
forked_thread_id = f"session_{uuid.uuid4()}"
fork_config = {"configurable": {"thread_id": forked_thread_id}}

# Initialize with alternative selection directly
fork_state = {
    "messages": [HumanMessage(content="How does the authenticate method work?")],
    "retrieved_candidates": [
        "src/auth/jwt_handler.py (def authenticate(token: str))",
        "src/auth/oauth_client.py (def authenticate(code: str))",
    ],
    "selected_entity": "src/auth/oauth_client.py (def authenticate(code: str))"
}

print(f"🔀 Running Forked Execution with OAuth target...")
forked_result = app.invoke(fork_state, fork_config)

print("\n=================== FORKED AGENT ANSWER (OAuth) ===================")
print(forked_result["final_response"])
print("===================================================================")


🔀 Running Forked Execution with OAuth target...

🔍 [STEP 1: RETRIEVAL] Searching symbols for query: 'how does the authenticate method work?'



💡 [GENERATION COMPLETE] Response length: 1691 chars

=================== FORKED AGENT ANSWER (OAuth) ===================
**`src/auth/oauth_client.py – `authenticate(code: str)`**

| Stage | What happens | Key details |
|-------|--------------|-------------|
| **1. Input** | Receives an OAuth *authorization code* (`code`). | The code is the short‑lived token returned by the authorization server after the user grants consent. |
| **2. Build token‑request** | Constructs a POST request to the provider’s *token endpoint*. | Payload includes:<br>• `grant_type=authorization_code`<br>• `code` (the argument)<br>• `redirect_uri` (must match the one used in the auth request)<br>• `client_id` & `client_secret` (from the app’s credentials). |
| **3. Send request** | Uses an HTTP client (e.g., `requests` or `httpx`) to POST the payload. | The request is sent over HTTPS; the client may set `Content-Type: application/x-www-form-urlencoded`. |
| **4. Handle response** | Parses the JSON response. | Exp